# UniAD walkthrough

End-to-end pipeline check on a single MVTec-AD category.

Steps:
1. Load config and dataset
2. Build the UniAD model
3. Run a forward pass on one batch
4. Inspect feature shapes, neighbor mask, anomaly map
5. (Optional) load a checkpoint and visualize anomaly heatmaps

This is for verifying the pipeline works end-to-end before launching a full training run from `scripts/train.py`.

In [ ]:
import os, sys, yaml
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(ROOT))

with open(ROOT / 'configs' / 'default.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['root'] = os.environ['UNIAD_DATA_ROOT']
cfg['data']['categories'] = ['bottle']
cfg['train']['batch_size'] = 4
cfg['eval']['batch_size'] = 4
cfg

In [ ]:
from src import UniAD, build_train_loader, build_eval_loader

train_loader = build_train_loader(cfg)
eval_loader = build_eval_loader(cfg)
len(train_loader.dataset), len(eval_loader.dataset)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UniAD(cfg).to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable params: {n_trainable/1e6:.2f}M')
print(f'feature channels: {model.backbone.feature_channels}')

In [ ]:
batch = next(iter(train_loader))
images = batch['image'].to(device)
model.train()
tokens, recon = model(images)
print('tokens:', tokens.shape, '| recon:', recon.shape)
print('mse:', (tokens - recon).pow(2).mean().item())

In [ ]:
nbr = model.transformer.neighbor_mask.squeeze().cpu().numpy()
plt.figure(figsize=(4,4))
plt.imshow(nbr, cmap='gray_r')
plt.title(f'neighbor mask ({cfg["model"]["neighbor_mask_size"]}x{cfg["model"]["neighbor_mask_size"]})')
plt.xlabel('key index'); plt.ylabel('query index')
plt.show()

In [ ]:
model.eval()
test_batch = next(iter(eval_loader))
imgs = test_batch['image'].to(device)
with torch.no_grad():
    amap = model.anomaly_map(imgs).cpu().numpy()
amap.shape, float(amap.min()), float(amap.max())

In [ ]:
fig, axes = plt.subplots(2, len(imgs), figsize=(4*len(imgs), 8))
for i in range(len(imgs)):
    img_np = imgs[i].cpu().numpy().transpose(1,2,0)
    img_np = (img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])).clip(0,1)
    axes[0, i].imshow(img_np); axes[0, i].set_title(f'label={int(test_batch["label"][i])}'); axes[0, i].axis('off')
    axes[1, i].imshow(amap[i], cmap='jet'); axes[1, i].set_title('anomaly map'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

Untrained model — anomaly maps are noise. After training (`scripts/train.py`) they should highlight defective regions for anomalous samples and stay flat for good samples.